# 07 — Modelos de Variable Dependiente Limitada
**Autores clave:** McFadden (1974) · Tobin (1958) · Heckman (1979) · Amemiya (1985) *Advanced Econometrics* · Greene (2012) Cap. 17–23

## ¿Por qué no usar OLS?
Cuando $Y$ es binaria, censurada o truncada, OLS produce:
- Predicciones fuera del rango $[0,1]$ para binarias
- Estimadores sesgados con censura/truncamiento

## Modelo Logit (McFadden, 1974)
$$P(Y=1|X) = \Lambda(X\beta) = \frac{e^{X\beta}}{1 + e^{X\beta}}$$
Estimación por **Maximum Likelihood**. Los coeficientes son log-odds.

## Modelo Probit
$$P(Y=1|X) = \Phi(X\beta)$$
Donde $\Phi$ es la CDF normal estándar. Los coeficientes se interpretan via **efectos marginales**.

## Modelo Tobit (Tobin, 1958)
Para datos censurados en 0 (ej. gasto, horas trabajadas, duración):
$$Y_i = \max(0, Y_i^*) \quad \text{donde} \quad Y_i^* = X_i\beta + u_i$$

## Modelo de Selección de Heckman (1979)
Corrige el **sesgo de selección muestral** en dos etapas:
1. Probit de selección → estimar $\lambda_i = \phi(Z_i\gamma) / \Phi(Z_i\gamma)$ (razón de Mills inversa)
2. OLS incluyendo $\lambda_i$ como regresor adicional

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit, Probit

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
n = 3000

# Covariables
educ   = np.random.randint(8, 20, n).astype(float)
exper  = np.clip(np.random.normal(10, 5, n), 0, 40)
female = np.random.binomial(1, 0.5, n).astype(float)
kids   = np.random.poisson(1.2, n).astype(float)

# Índice latente para empleo
idx_employ = -1.0 + 0.15*educ + 0.03*exper - 0.30*female - 0.20*kids + np.random.normal(0, 1, n)
employed   = (idx_employ > 0).astype(float)

# Salario (solo observado si emplea)
u_wage  = np.random.normal(0, 0.5, n)
log_wage_star = 1.0 + 0.10*educ + 0.02*exper - 0.20*female + u_wage
log_wage = np.where(employed == 1, log_wage_star, np.nan)

df = pd.DataFrame({
    'log_wage': log_wage, 'employed': employed,
    'educ': educ, 'exper': exper, 'female': female, 'kids': kids,
})
print(f'Tasa de empleo: {employed.mean():.1%}')
print(f'Obs con salario observable: {employed.sum():.0f}')

## 1 — Logit y Probit: Efectos Marginales

In [ ]:
X_vars = ['educ', 'exper', 'female', 'kids']
X = sm.add_constant(df[X_vars])
y = df['employed']

# OLS (LPM)
mod_lpm   = sm.OLS(y, X).fit(cov_type='HC3')

# Logit
mod_logit = Logit(y, X).fit(disp=False)

# Probit
mod_probit = Probit(y, X).fit(disp=False)

# Efectos marginales en la media (MEM)
me_logit  = mod_logit.get_margeff()
me_probit = mod_probit.get_margeff()

print('Probabilidad Lineal (LPM) vs Logit vs Probit — Efectos Marginales en la Media')
print(f'{"Variable":<10} {"LPM":>10} {"Logit ME":>12} {"Probit ME":>12}')
print('─' * 48)
for var in X_vars:
    lpm   = mod_lpm.params[var]
    logit = me_logit.margeff[list(me_logit.summary_frame().index).index(var)]
    probit= me_probit.margeff[list(me_probit.summary_frame().index).index(var)]
    print(f'{var:<10} {lpm:>10.4f} {logit:>12.4f} {probit:>12.4f}')

# Visualizar curva logística vs probit
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

educ_range = np.linspace(8, 20, 100)
X_pred = pd.DataFrame({'const': 1, 'educ': educ_range,
                        'exper': np.full(100, exper.mean()),
                        'female': np.zeros(100), 'kids': np.zeros(100)})

p_logit  = mod_logit.predict(X_pred)
p_probit = mod_probit.predict(X_pred)
p_lpm    = mod_lpm.predict(X_pred)

axes[0].plot(educ_range, p_logit,  color='#f72585', linewidth=2.5, label='Logit')
axes[0].plot(educ_range, p_probit, color='#4361ee', linewidth=2.5, label='Probit', linestyle='--')
axes[0].plot(educ_range, p_lpm,    color='#06d6a0', linewidth=2,   label='LPM', linestyle=':')
axes[0].axhline(0, color='#aaa', linewidth=1); axes[0].axhline(1, color='#aaa', linewidth=1)
axes[0].set_xlabel('Educación'); axes[0].set_ylabel('P(employed=1)')
axes[0].set_title('LPM vs Logit vs Probit\n(exper=media, female=0, kids=0)')
axes[0].legend(fontsize=9)

# Comparación de predicciones
y_hat_logit = mod_logit.predict(X)
axes[1].scatter(y_hat_logit, mod_probit.predict(X), s=5, alpha=0.3, color='#4361ee')
axes[1].plot([0,1],[0,1],'k--', linewidth=2)
axes[1].set_xlabel('P̂ Logit'); axes[1].set_ylabel('P̂ Probit')
axes[1].set_title('Predicciones Logit vs Probit\n(casi idénticas en la práctica)')

plt.tight_layout()
plt.show()

## 2 — Corrección de Heckman (1979): Sesgo de Selección

In [ ]:
from scipy.stats import norm

df_emp = df[df['employed'] == 1].copy()

# OLS ingenuo (solo sobre empleados — sesgado)
X_wage = sm.add_constant(df_emp[['educ', 'exper', 'female']])
mod_ols_naive = sm.OLS(df_emp['log_wage'], X_wage).fit()

# Heckman dos etapas (Heckit)
# Etapa 1: Probit de selección (incluye 'kids' como exclusión)
X_select = sm.add_constant(df[['educ', 'exper', 'female', 'kids']])
probit_select = Probit(df['employed'], X_select).fit(disp=False)

# Razón de Mills Inversa (IMR)
xb       = probit_select.predict(X_select, linear=True)
df['IMR'] = norm.pdf(xb) / norm.cdf(xb)

# Etapa 2: OLS incluyendo IMR
df_emp2 = df[df['employed'] == 1].copy()
X_heck  = sm.add_constant(df_emp2[['educ', 'exper', 'female', 'IMR']])
mod_heck = sm.OLS(df_emp2['log_wage'], X_heck).fit()

print('Corrección de Heckman (1979) — Sesgo de Selección Muestral')
print(f'{"Variable":<10} {"OLS naïve":>12} {"Heckman":>12} {"Verdadero":>12}')
print('─' * 50)
true_vals = {'educ': 0.10, 'exper': 0.02, 'female': -0.20}
for var in ['educ', 'exper', 'female']:
    print(f'{var:<10} {mod_ols_naive.params[var]:>12.4f} {mod_heck.params[var]:>12.4f} {true_vals[var]:>12.4f}')
print(f'{"IMR (λ)":<10} {"—":>12} {mod_heck.params["IMR"]:>12.4f}  (rho·sigma si significativo → selección)')
print()
print(f'IMR coef p-value: {mod_heck.pvalues["IMR"]:.4f}  '
      f'{"→ selección significativa" if mod_heck.pvalues["IMR"] < 0.05 else "→ selección no significativa"}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['IMR'], bins=40, color='#4361ee', alpha=0.8)
axes[0].set_xlabel('Razón de Mills Inversa (λ_i)')
axes[0].set_title('Distribución de λ_i\n→ Alta = alta probabilidad de no emplearse')

axes[1].scatter(df_emp2['educ'], df_emp2['log_wage'], s=6, alpha=0.3, color='#adb5bd', label='Datos')
educ_r = np.linspace(8, 20, 100)
for name, coef_educ, coef_int, color in [
    ('OLS naïve', mod_ols_naive.params['educ'], mod_ols_naive.params['const'], '#f72585'),
    ('Heckman',   mod_heck.params['educ'],      mod_heck.params['const'],      '#4361ee'),
]:
    axes[1].plot(educ_r, coef_int + coef_educ*educ_r, linewidth=2.5, color=color,
                 label=f'{name}: β={coef_educ:.3f}')
axes[1].axline((0, 1.0 + 0.10*0), slope=0.10, color='#06d6a0', linewidth=2,
               linestyle='--', label='Verdadero: β=0.10')
axes[1].set_xlabel('Educación'); axes[1].set_ylabel('log(salario)')
axes[1].set_title('OLS naïve vs Heckman')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Resumen

| Modelo | Cuándo usar | Estimación | Interpretación |
|---|---|---|---|
| LPM | Efectos marginales aproximados | OLS | Cambio en probabilidad |
| Logit | Y binaria, efectos no lineales | MLE | Log-odds → efectos marginales |
| Probit | Y binaria, base en utilidad latente | MLE | Efectos marginales |
| Tobit | Y censurada en 0 | MLE | E[Y*] e E[Y|Y>0] |
| Heckman | Muestra seleccionada | 2 etapas | Correige λ (IMR) |

**Referencias:**
- Tobin, J. (1958). Estimation of relationships for limited dependent variables. *Econometrica*, 26(1).
- McFadden, D. (1974). Conditional logit analysis of qualitative choice behavior. *Frontiers in Econometrics*. Academic Press.
- Heckman, J.J. (1979). Sample selection bias as a specification error. *Econometrica*, 47(1).
- Amemiya, T. (1985). *Advanced Econometrics*. Harvard UP.
- Greene, W.H. (2012). *Econometric Analysis* (7th ed.), Cap. 17–23.

**Siguiente:** `08_time_series_var.ipynb`